# 집계 후보 합집합: 다른 세 영상 검증

`0936`에서 후보 합집합이 7회 확정·1회 미확정으로 개선돼, `0188·0518·0642`에서 같은 방법을 시험했습니다. 각 영상의 기존 0.5초 실행에서 사용한 질문 해석과 `gpt-4o` 모델을 고정하고 0.25초 전체 탐색·후보 검증을 다시 실행했습니다. 합집합은 두 실행에서 이미 검증한 전·후 장면을 재사용하고, 영상 간이 아닌 **같은 영상 내 실행 간 강아지 ID와 리셋**을 다시 확인했습니다.

`0188`은 사용자가 2회 구간을 확인했습니다. `0518·0642`의 사건별 사람 시각은 프레임 잠정 검토라 성능 지표도 잠정입니다. 후보가 기준 구간 ±0.75초 안에 있다는 사실만으로 행동 진입이나 개체 ID가 입증되지는 않습니다.

## 결과

| 영상 | 사람 기준 | 0.5초 확정 | 0.25초 확정 | 후보 합집합 확정 | 합집합 미확정 |
|---|---:|---:|---:|---:|---:|
| `0188` 울타리 | 사용자 2 | 2 | 2 | **2** | 7 |
| `0518` 소파 | 잠정 2 | 2 | 2 | **2** | 0 |
| `0642` 분수 | 잠정 2 | 1 | 1 | **1** | 7 |

`0188`과 `0518`은 합집합으로 횟수가 부풀지 않았습니다. `0642`는 기존 실행에서 첫 번째 입수(약 2.0초), 새 실행에서 두 번째 입수(후보 6.5초, 검증된 전·후 장면 4.0→5.25초)를 각각 확정했지만, 두 실행의 후보를 함께 놓고 자동으로 개체 ID를 다시 연결할 때 두 번째 강아지를 `unknown`으로 남겨 최종 1회입니다.

In [1]:
import json
from pathlib import Path
root = Path.cwd()
if root.name == "notebooks": root = root.parent
report_path = root / "notebooks/smarthome-bench-evidence/new-pipeline/ablation-cross-video/offline-audit.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
for case in report["cases"]:
    print(f'{case["video_id"]}: 0.5초 {case["step050_confirmed"]}회, '
          f'0.25초 {case["step025_confirmed"]}회, 합집합 {case["union_confirmed"]}회 '
          f'(미확정 {case["union_uncertain"]})')

smartbench_0188: 0.5초 2회, 0.25초 2회, 합집합 2회 (미확정 7)
smartbench_0518: 0.5초 2회, 0.25초 2회, 합집합 2회 (미확정 0)
smartbench_0642: 0.5초 1회, 0.25초 1회, 합집합 1회 (미확정 7)

## 후보 탐색과 최종 집계를 나누어 보기

| 영상 | 0.5초 후보 제안 | 0.25초 후보 제안 | 각 탐색에서 시각이 맞는 사건 | 최종 병목 |
|---|---:|---:|---:|---|
| `0188` | 10 | 9 | 각각 2/2 | 지속 울타리 자세의 여러 후보를 중복으로 걸러야 함 |
| `0518` | 6 | 4 | 각각 2/2 | 검은·흰 강아지를 서로 다른 ID로 유지해야 함 |
| `0642` | 4 | 8 | 각각 2/2 | 늦은 후보 시각과 비슷한 강아지의 ID 확인 |

`0642`의 0.25초 탐색은 8.0~14.5초에도 후보를 냈습니다. 저장된 응답 일부는 이미 물속에 계속 있는 장면이라고 스스로 설명합니다. 후보 제안이 많아져도 새 입수가 늘었다는 뜻은 아닙니다. 0.5초 후보 6.25초의 첫 검증 프레임(5.0초)은 강아지가 이미 물에 부분적으로 들어가 있어 ‘물 밖’ 근거로 부족했습니다. 0.25초 실행은 더 넓은 재검사에서 4.0초 물 밖 → 5.25초 물 안을 찾았습니다.

In [ ]:
case = next(c for c in report["cases"] if c["video_id"] == "smartbench_0642")
print("0642 기존 후보:", case["step050_scan"]["candidate_times"])
print("0642 새 후보:", case["step025_scan"]["candidate_times"])
print("합집합 확정/미확정:", case["union_confirmed"], case["union_uncertain"])
print("확대 프레임 ID 확인:", case["cropped_identity_check"]["response"])

## `0642` 확대 프레임 재확인

자동 ID 연결은 두 번째 입수 강아지를 `unknown`으로 남겼습니다. 2.0·4.0·4.5·5.0·5.25초의 **사람이 고른 분수 주변 확대 영역**을 별도로 모델에 보여주자, 4.0초에 물속 강아지와 오른쪽 바위의 강아지가 동시에 보이고 바위 쪽 강아지가 이후 들어간다는 근거로 서로 다른 개체라고 응답했습니다. 원본 전체 프레임만 사용한 확인은 미확정이었습니다.

이 확대 실험은 원인 진단이며 **완전 자동 집계 결과가 아닙니다.** 자동 결과는 계속 1회 확정·미확정 후보입니다. 일반화하려면 강아지 위치를 자동으로 찾고, 같은 프레임에서 두 개체가 동시에 보이는 장면을 ID 검증에 넣어야 합니다.

## 원본 영상 확인

로컬 `data/videos/`에 파일이 있으면 아래 셀로 재생합니다. `0642`의 4~5.25초에는 물속 강아지와 오른쪽 바위에서 들어오는 강아지를 함께 보세요. 저장된 비교 JSON은 영상·API 키 없이 읽을 수 있습니다.

In [ ]:
for video_id in ("0188", "0518", "0642"):
    video = root / "data/videos" / f"smartbench_{video_id}.mp4"
    print(video_id, video if video.exists() else "로컬 영상 없음")
    if video.exists():
        try:
            from IPython.display import Video, display
        except ImportError:
            print("Jupyter/IPython에서 재생 가능")
        else:
            display(Video(filename=str(video), embed=False, html_attributes="controls"))

## 재실행·해석

각 영상의 고정 질문 해석은 `ablation-cross-video/smartbench_*/fixed-intent.json`에 있습니다. API 호출이 필요한 재실행 명령 예시는 아래와 같습니다. 두 번째 명령은 0.5초·0.25초 결과가 모두 있을 때 실행합니다.

```bash
.venv/bin/python scripts/validate_video_aggregate.py --ids 0642 --scan-model gpt-4o --verify-model gpt-4o --coarse-step 0.25 --intent-file notebooks/smarthome-bench-evidence/new-pipeline/ablation-cross-video/smartbench_0642/fixed-intent.json --output notebooks/smarthome-bench-evidence/new-pipeline/ablation-cross-video/smartbench_0642/step025
.venv/bin/python scripts/validate_cross_video_union.py --ids 0188 0518 0642
python3 scripts/audit_cross_video_candidates.py
```

이번 세 영상에서 후보 합집합이 **새 정답 사건을 추가로 확정한 사례는 없었습니다.** `0936`의 개선을 모든 행동에 일반화할 근거는 아직 없습니다. `0642`는 촘촘한 탐색보다 작은·비슷한 강아지를 분리하는 시각 근거가 더 중요한 실패 사례입니다. 이 노트북은 로컬 실험이며 운영 `/query`나 worker 배포 결과가 아닙니다.